In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# pasta onde ficam as imagens e os CSVs gerados
PASTA_RESULTADOS = Path("Resultados")
PASTA_RESULTADOS.mkdir(exist_ok=True)

In [2]:
pasta_dados = Path("DadosBrutos") if Path("DadosBrutos").exists() else Path("..") / "DadosBrutos"
caminho_arquivo = pasta_dados / "V_EXPORTACAO_.csv"
df = pd.read_csv(caminho_arquivo, sep=';', encoding='utf-8-sig')


print(df.head())




    Ano  Código NCM                                      Descrição NCM  \
0  2025    17019900  Outros açúcares de cana, beterraba, sacarose q...   
1  2025    17019900  Outros açúcares de cana, beterraba, sacarose q...   
2  2025    17019900  Outros açúcares de cana, beterraba, sacarose q...   
3  2025    17019900  Outros açúcares de cana, beterraba, sacarose q...   
4  2025    21039029           Outros condimentos e temperos, compostos   

                Países  Valor US$ FOB  Quilograma Líquido  
0               Angola      122025306           260663908  
1           Cabo Verde        5050904            11062770  
2  São Tomé e Príncipe        1383827             2970000  
3         Guiné-Bissau         870370             1772000  
4               Angola         851720              254456  


## 1. Carregar o extrato

In [3]:
pasta_dados = Path("DadosBrutos") if Path("DadosBrutos").exists() else Path("..") / "DadosBrutos"
caminho_arquivo = pasta_dados / "V_EXPORTACAO_.csv"
df = pd.read_csv(caminho_arquivo, sep=';', encoding='utf-8-sig')

colunas_texto = ['Países', 'Descrição NCM']
for col in colunas_texto:
    df[col] = df[col].str.replace(r'[\r\n]+', ' ', regex=True).str.strip()

# período analisado
ANO_INICIAL = 2015
ANO_FINAL = 2025

df = df[(df['Ano'] >= ANO_INICIAL) & (df['Ano'] <= ANO_FINAL)]
df = df.reset_index(drop=True)     # o laço da seção 3 usa df.loc[i], então o índice precisa ir de 0 a n-1
print("anos usados:    ", ANO_INICIAL, "a", ANO_FINAL)

print(df.shape)
df.head(10)

anos usados:     2015 a 2025
(893, 6)


,Ano,Código NCM,Descrição NCM,Países,Valor US$ FOB,Quilograma Líquido
0,2025,17019900,"Outros açúcares de cana, beterraba, sacarose q...",Angola,122025306,260663908
1,2025,17019900,"Outros açúcares de cana, beterraba, sacarose q...",Cabo Verde,5050904,11062770
2,2025,17019900,"Outros açúcares de cana, beterraba, sacarose q...",São Tomé e Príncipe,1383827,2970000
3,2025,17019900,"Outros açúcares de cana, beterraba, sacarose q...",Guiné-Bissau,870370,1772000
4,2025,21039029,"Outros condimentos e temperos, compostos",Angola,851720,254456
5,2025,19019090,"Outras preparações alimentícias de farinhas, e...",Cabo Verde,459115,263707
6,2025,17019900,"Outros açúcares de cana, beterraba, sacarose q...",Moçambique,414674,833000
7,2025,17049020,"Caramelos, confeitos, dropes, pastilhas, e pro...",Angola,376909,251150
8,2025,20098990,"Sucos (sumo) de outras frutas, não fermentado,...",Cabo Verde,226939,171201
9,2025,20098990,"Sucos (sumo) de outras frutas, não fermentado,...",Moçambique,165000,150000


## 2. Corrigir o Código NCM

O NCM tem 8 dígitos e alguns começam com `0`. Como o pandas lê a coluna como número,
esse zero se perde.

In [4]:
print("antes:")
print(df['Código NCM'].astype(str).str.len().value_counts())

df['Código NCM'] = df['Código NCM'].astype(str).str.zfill(8)

print()
print("depois:")
print(df['Código NCM'].str.len().value_counts())
print("NCMs distintos no extrato:", df['Código NCM'].nunique())

antes:
Código NCM
8    774
7    119
Name: count, dtype: int64

depois:
Código NCM
8    893
Name: count, dtype: int64
NCMs distintos no extrato: 50


## 3. Montar a tabela

Uma linha por par (NCM, país), uma coluna por ano.

Os valores são os do ComexStat, sem nenhum filtro. O ComexStat não permite identificar,
apenas pelo valor, se uma operação foi venda comercial, amostra ou pequena remessa —
estabelecer um valor de corte seria decisão arbitrária da pesquisa e não regra da fonte.
Zero significa ausência de registro naquele ano.

In [5]:
primeiro_ano = ANO_INICIAL
ultimo_ano = ANO_FINAL
anos_base = list(range(primeiro_ano, ultimo_ano + 1))

paises_palop = ['Angola', 'Cabo Verde', 'Guiné-Bissau', 'Moçambique', 'São Tomé e Príncipe']

ncms = sorted(df['Código NCM'].unique())


# 1) cria uma lista de zeros (um por ano) para cada par (NCM, país)
series = {}
for ncm in ncms:
    for pais in paises_palop:
        zeros = []
        for ano in anos_base:
            zeros.append(0)
        series[(ncm, pais)] = zeros


# 2) percorre o   e soma cada valor na posição do ano
for i in range(len(df)):
    ncm = df.loc[i, 'Código NCM']
    pais = df.loc[i, 'Países']
    ano = df.loc[i, 'Ano']
    valor = df.loc[i, 'Valor US$ FOB']

    if pais in paises_palop:
        posicao = anos_base.index(ano)
        series[(ncm, pais)][posicao] = series[(ncm, pais)][posicao] + valor


# 3) transforma o dicionário em DataFrame
linhas = []
chaves = []
for chave in series:
    chaves.append(chave)
    linhas.append(series[chave])

indice = pd.MultiIndex.from_tuples(chaves, names=['Código NCM', 'Países'])
tabela = pd.DataFrame(linhas, columns=anos_base, index=indice)


print("Anos:", primeiro_ano, "a", ultimo_ano)
print("NCMs:", len(ncms))
print("Linhas esperadas:", len(ncms) * len(paises_palop))
print("Linhas na tabela:", tabela.shape[0])
tabela.head(20)

Anos:

 2015 a 2025
NCMs: 50
Linhas esperadas: 250
Linhas na tabela: 250


2015  2016  2017  2018  2019   2020  2021  \
Código NCM Países                                                           
07129090   Angola                731     0     0     0     0    247     0   
           Cabo Verde              0     0     0     0     0      0     0   
           Guiné-Bissau            0     0     0     0     0      0     0   
           Moçambique              0     0     0     0     0      0     0   
           São Tomé e Príncipe     0     0     0     0     0      0     0   
08011100   Angola               1736  4315  5336  4938  5691  27942  6026   
           Cabo Verde              0  1371   220   690  2826   1006     0   
           Guiné-Bissau            0     0     0     0     0      0     0   
           Moçambique              0     0     0     0     0     78     0   
           São Tomé e Príncipe     0     0   162     0     0      0     0   
09041100   Angola                  0     0   320     0     0     39     0   
           Cabo Verde              0     0     0     0     0      0     0   
           Guiné-Bissau            0     0     0     0     0      0     0   
           Moçambique              0     0     0     0     0      0     0   
           São Tomé e Príncipe     0     0     0     0     0      0     0   
09041200   Angola               7691     0     0     0     0   1547  1708   
           Cabo Verde              0     0     0     0     0      0     0   
           Guiné-Bissau            0     0     0     0     0      0     0   
           Moçambique              0     0     0     0     0      0     0   
           São Tomé e Príncipe     0     0   194     0     0      0     0   

                                 2022  2023  2024  2025  
Código NCM Países                                        
07129090   Angola                 292     0  2985     0  
           Cabo Verde               0     0     0     0  
           Guiné-Bissau             0     0    19    22  
           Moçambique               0     0     0    99  
           São Tomé e Príncipe      0     0     0     0  
08011100   Angola               30087     1  4452  6701  
           Cabo Verde               0  1534   368   206  
           Guiné-Bissau             0     0    50     0  
           Moçambique               0     0     0     0  
           São Tomé e Príncipe      0     0     0     0  
09041100   Angola                 301     0     0    14  
           Cabo Verde               0     1     0     0  
           Guiné-Bissau             0     0    22     0  
           Moçambique               0     0     0     0  
           São Tomé e Príncipe      0     0     0     0  
09041200   Angola                5372  2140    59     0  
           Cabo Verde               0   198     0     0  
           Guiné-Bissau             0     0    86    42  
           Moçambique               0     0     0     0  
           São Tomé e Príncipe      0     0     0    99

## 4. Continuidade

Exportou em **todos** os anos da série.
Ordenação: pelo valor total exportado.

In [6]:
def eh_continua(valores):
    for valor in valores:
        if valor == 0:
            return False
    return True

In [7]:
def calcular_valor_total(valores):
    total = 0
    for valor in valores:
        total = total + valor
    return total

In [8]:
def pegar_valor_total(item):
    return item['valor_total']


def ordenar_continuidades(tabela):
    # ordena pelo valor total exportado, do maior para o menor
    lista = []

    for (ncm, pais), linha in tabela.iterrows():
        valores = linha.tolist()
        if eh_continua(valores):
            lista.append({
                'ncm': ncm,
                'pais': pais,
                'valor_total': calcular_valor_total(valores),
            })

    lista.sort(key=pegar_valor_total, reverse=True)
    return lista


continuidades = ordenar_continuidades(tabela)

print("=== CONTINUIDADE:", len(continuidades), "pares ===")
for posicao in range(len(continuidades)):
    item = continuidades[posicao]
    print(str(posicao + 1).rjust(3), item['ncm'], item['pais'].ljust(22),
          "US$", format(item['valor_total'], ',d').rjust(15))

=== CONTINUIDADE: 31 pares ===
  1 17019900 Angola                 US$   1,075,985,947
  2 17019900 Cabo Verde             US$      56,901,166
  3 17049020 Angola                 US$      13,425,620
  4 17019900 São Tomé e Príncipe    US$       9,802,227
  5 17019900 Guiné-Bissau           US$       8,443,523
  6 21069010 Cabo Verde             US$       6,589,809
  7 18069000 Angola                 US$       2,389,389
  8 21039021 Angola                 US$       2,039,893
  9 17049020 Moçambique             US$       1,967,014
 10 17049020 Cabo Verde             US$       1,792,538
 11 20098990 Cabo Verde             US$       1,113,508
 12 17049020 São Tomé e Príncipe    US$         812,796
 13 20098990 Angola                 US$         640,970
 14 21069029 Angola                 US$         637,537
 15 19023000 Cabo Verde             US$         619,105
 16 21069029 Cabo Verde             US$         548,645
 17 18069000 Cabo Verde             US$         521,305
 18 19041000 Ango

## 5. Surgimento

Não exportava no primeiro ano, começou em algum momento e seguiu sem parar até o último.
Ordenação: por quantos anos já dura; em caso de empate, pelo valor total.

In [9]:
def contar_anos_ativos(valores):
    total = 0
    for valor in valores:
        if valor > 0:
            total = total + 1
    return total

In [10]:
ANOS_MINIMOS_SURGIMENTO = 3


def eh_surgimento(valores):
    # no primeiro ano não pode ter exportado
    if valores[0] > 0:
        return False

    # no último ano tem que ter exportado
    if valores[-1] == 0:
        return False

    # depois que começou, não pode ter parado em nenhum ano
    ja_comecou = False
    for valor in valores:
        if valor > 0:
            ja_comecou = True
        elif ja_comecou:
            return False

    # precisa ter comprado em pelo menos 3 anos para ser considerado surgimento
    if contar_anos_ativos(valores) < ANOS_MINIMOS_SURGIMENTO:
        return False

    return True

In [11]:
def pegar_anos_e_valor(item):
    # ordena primeiro por anos ativos; empatou, decide pelo valor total
    return (item['anos_ativos'], item['valor_total'])


def ordenar_surgimentos(tabela):
    # ordena por quantos anos já dura; em caso de empate, pelo valor total
    lista = []

    for (ncm, pais), linha in tabela.iterrows():
        valores = linha.tolist()
        if eh_surgimento(valores):
            lista.append({
                'ncm': ncm,
                'pais': pais,
                'anos_ativos': contar_anos_ativos(valores),
                'valor_total': calcular_valor_total(valores),
            })

    lista.sort(key=pegar_anos_e_valor, reverse=True)
    return lista


surgimentos = ordenar_surgimentos(tabela)

print("=== SURGIMENTO:", len(surgimentos), "pares ===")
for posicao in range(len(surgimentos)):
    item = surgimentos[posicao]
    print(str(posicao + 1).rjust(3), item['ncm'], item['pais'].ljust(22),
          str(item['anos_ativos']).rjust(2), "anos ",
          "US$", format(item['valor_total'], ',d').rjust(12))

=== SURGIMENTO: 2 pares ===
  1 20098990 Moçambique              7 anos  US$      907,467
  2 18050000 Angola                  7 anos  US$      135,385


## 6. Descontinuidade

Exportava desde o primeiro ano, parou em algum momento e não voltou.
Ordenação: por quantos anos ficou ativo; em caso de empate, pelo valor total.

In [12]:
def eh_descontinuidade(valores):
    # no primeiro ano tem que ter exportado
    if valores[0] == 0:
        return False

    # no último ano não pode ter exportado
    if valores[-1] > 0:
        return False

    # depois que parou, não pode ter voltado
    ja_parou = False
    for valor in valores:
        if valor == 0:
            ja_parou = True
        elif ja_parou:
            return False

    return True

In [13]:
def ordenar_descontinuidades(tabela):
    # ordena por quantos anos ficou ativo; em caso de empate, pelo valor total
    lista = []

    for (ncm, pais), linha in tabela.iterrows():
        valores = linha.tolist()
        if eh_descontinuidade(valores):
            lista.append({
                'ncm': ncm,
                'pais': pais,
                'anos_ativos': contar_anos_ativos(valores),
                'valor_total': calcular_valor_total(valores),
            })

    lista.sort(key=pegar_anos_e_valor, reverse=True)
    return lista


descontinuidades = ordenar_descontinuidades(tabela)

print("=== DESCONTINUIDADE:", len(descontinuidades), "pares ===")
for posicao in range(len(descontinuidades)):
    item = descontinuidades[posicao]
    print(str(posicao + 1).rjust(3), item['ncm'], item['pais'].ljust(22),
          str(item['anos_ativos']).rjust(2), "anos ",
          "US$", format(item['valor_total'], ',d').rjust(12))

=== DESCONTINUIDADE: 8 pares ===
  1 11041200 Angola                 10 anos  US$       31,217
  2 09109900 Angola                  9 anos  US$        9,809
  3 12040090 Angola                  9 anos  US$        1,696
  4 21041011 Angola                  8 anos  US$    3,851,733
  5 19019090 São Tomé e Príncipe     3 anos  US$        4,130
  6 21041019 Cabo Verde              1 anos  US$        2,434
  7 20059900 Moçambique              1 anos  US$          912
  8 28369913 Angola                  1 anos  US$          174


## 7. Intermitente

É o caso que sobra: exportou em algum ano, mas não se encaixa em nenhum dos três
anteriores — vai e volta.

Ordenação: pela **maior sequência seguida** de anos parados (quanto menor, melhor);
em caso de empate, pelo valor total. Uma série que parou três anos seguidos é pior
que uma que parou três anos alternados, porque a relação comercial chegou a se romper.

In [14]:
def eh_intermitente(valores):
    if eh_continua(valores):
        return False
    if eh_surgimento(valores):
        return False
    if eh_descontinuidade(valores):
        return False
    if contar_anos_ativos(valores) == 0:
        return False

    return True

In [15]:
def maior_sequencia_parada(valores):
    # maior número de anos parados em seguida
    maior = 0
    atual = 0

    for valor in valores:
        if valor == 0:
            atual = atual + 1
            if atual > maior:
                maior = atual
        else:
            atual = 0          # exportou: zera a contagem da sequência

    return maior

In [16]:
def pegar_parada_e_valor(item):
    return (item['maior_parada'], -item['valor_total'])


def ordenar_intermitentes(tabela):
    lista = []

    for (ncm, pais), linha in tabela.iterrows():
        valores = linha.tolist()
        if eh_intermitente(valores):
            lista.append({
                'ncm': ncm,
                'pais': pais,
                'maior_parada': maior_sequencia_parada(valores),
                'valor_total': calcular_valor_total(valores),
            })

    lista.sort(key=pegar_parada_e_valor)
    return lista


intermitentes = ordenar_intermitentes(tabela)

print("=== INTERMITENTE:", len(intermitentes), "pares ===")
for posicao in range(len(intermitentes)):
    item = intermitentes[posicao]
    print(str(posicao + 1).rjust(3), item['ncm'], item['pais'].ljust(22),
          str(item['maior_parada']).rjust(2), "anos seguidos parado ",
          "US$", format(item['valor_total'], ',d').rjust(12))

=== INTERMITENTE: 137 pares ===
  1 20059900 Angola                  1 anos seguidos parado  US$    4,374,896
  2 19021900 Angola                  1 anos seguidos parado  US$      976,861
  3 19019090 Cabo Verde              1 anos seguidos parado  US$      943,392
  4 12119090 Moçambique              1 anos seguidos parado  US$      328,479
  5 12119090 Angola                  1 anos seguidos parado  US$      119,634
  6 21021090 Angola                  1 anos seguidos parado  US$      105,656
  7 19030000 Angola                  1 anos seguidos parado  US$       80,911
  8 20059900 Cabo Verde              1 anos seguidos parado  US$       75,084
  9 19041000 Cabo Verde              1 anos seguidos parado  US$       70,117
 10 20052000 Cabo Verde              1 anos seguidos parado  US$       55,803
 11 21069021 Angola                  1 anos seguidos parado  US$       47,517
 12 11081200 Angola                  1 anos seguidos parado  US$       45,238
 13 19030000 Cabo Verde         

## 8. Conferência

Cada par tem que cair em exatamente um caso. A soma dos cinco precisa dar o número de pares
da tabela (NCMs × 5 países).

In [17]:
nunca_exportou = 0
for (ncm, pais), linha in tabela.iterrows():
    if contar_anos_ativos(linha.tolist()) == 0:
        nunca_exportou = nunca_exportou + 1

total_esperado = len(ncms) * len(paises_palop)
total_casos = (len(continuidades) + len(surgimentos)
               + len(descontinuidades) + len(intermitentes) + nunca_exportou)

print("continuidade    ", len(continuidades))
print("surgimento      ", len(surgimentos))
print("descontinuidade ", len(descontinuidades))
print("intermitente    ", len(intermitentes))
print("nunca exportou  ", nunca_exportou)
print()
print("TOTAL           ", total_casos, "(tem que dar", total_esperado, ")")
print("conferência ok: ", total_casos == total_esperado)

continuidade    

 31
surgimento       2
descontinuidade  8
intermitente     137
nunca exportou   72

TOTAL            250 (tem que dar 250 )
conferência ok:  True


## 9. Tabela cruzada: casos por país

In [18]:
def classificar(valores):
    if eh_continua(valores):
        return "continuidade"
    if eh_surgimento(valores):
        return "surgimento"
    if eh_descontinuidade(valores):
        return "descontinuidade"
    if eh_intermitente(valores):
        return "intermitente"
    return "nunca exportou"


descricoes = df.groupby('Código NCM')['Descrição NCM'].first()

linhas_resultado = []
for (ncm, pais), linha in tabela.iterrows():
    valores = linha.tolist()
    linhas_resultado.append({
        'Código NCM': ncm,
        'Descrição NCM': descricoes[ncm],
        'Países': pais,
        'caso': classificar(valores),
        'anos_ativos': contar_anos_ativos(valores),
        'maior_parada': maior_sequencia_parada(valores),
        'valor_total': calcular_valor_total(valores),
    })

resultado = pd.DataFrame(linhas_resultado)

print(resultado['caso'].value_counts().to_string())
print()
pd.crosstab(resultado['Países'], resultado['caso'])

caso
intermitente       137
nunca exportou      72
continuidade        31
descontinuidade      8
surgimento           2

caso,continuidade,descontinuidade,intermitente,nunca exportou,surgimento
Países,,,,,
Angola,13,5,31,0,1
Cabo Verde,11,1,26,12,0
Guiné-Bissau,1,0,27,22,0
Moçambique,4,1,32,12,1
São Tomé e Príncipe,2,1,21,26,0


## 10. Pesos e ranking final

Os casos são exclusivos: cada par (NCM, país) cai em exatamente um deles. Por isso o peso
não entra numa soma — ele diz **qual caso ganha de qual**.

| Peso | Caso | O que diz sobre o mercado |
|---|---|---|
| 35 | continuidade | demanda comprovada, comprou em todos os anos |
| 30 | surgimento | demanda crescendo, entrou e ficou |
| 20 | intermitente | demanda instável, compra e para |
| 15 | descontinuidade | demanda perdida, comprava e parou |
| 0 | nunca exportou | sem evidência de demanda |

A ordenação final tem dois níveis:

1. **peso** — define o grupo (quem é continuidade vem antes de quem é surgimento);
2. **posição dentro do caso** — define a ordem entre os pares do mesmo grupo, usando o
   critério de cada caso (seções 4 a 7).

O segundo nível é o que evita empate: num produto onde três países são continuidade, é o
valor total que decide quem é 1º, 2º e 3º.

In [19]:
PESO_BASE = {
    "continuidade":    35,
    "surgimento":      30,
    "intermitente":    20,
    "descontinuidade": 15,
    "nunca exportou":   0,
}


# 1) pega as listas já ordenadas de cada caso (seções 4 a 7)
ordenadas = {
    "continuidade":    ordenar_continuidades(tabela),
    "surgimento":      ordenar_surgimentos(tabela),
    "descontinuidade": ordenar_descontinuidades(tabela),
    "intermitente":    ordenar_intermitentes(tabela),
}


# 2) guarda a posição de cada par dentro do seu caso (1 = melhor do caso)
posicao_no_caso = {}
for caso in ordenadas:
    lista = ordenadas[caso]
    for i in range(len(lista)):
        chave = (lista[i]['ncm'], lista[i]['pais'])
        posicao_no_caso[chave] = i + 1


# 3) escreve o peso e a posição no caso em cada linha do resultado
pesos = []
posicoes = []
for i in range(len(resultado)):
    ncm  = resultado.loc[i, 'Código NCM']
    pais = resultado.loc[i, 'Países']
    caso = resultado.loc[i, 'caso']

    pesos.append(PESO_BASE[caso])

    chave = (ncm, pais)
    if chave in posicao_no_caso:
        posicoes.append(posicao_no_caso[chave])
    else:
        posicoes.append(999)      # nunca exportou: não tem posição própria

resultado['peso'] = pesos
resultado['posicao_no_caso'] = posicoes


# 4) ordena cada NCM: primeiro pelo peso (maior antes),
#    depois pela posição dentro do caso (menor antes)
ranking = resultado.sort_values(['Código NCM', 'peso', 'posicao_no_caso'],
                                ascending=[True, False, True])

ranking['posicao'] = ranking.groupby('Código NCM').cumcount() + 1

print("linhas no ranking:", len(ranking))
print()
print("primeiros lugares por país:")
print(ranking[ranking['posicao'] == 1]['Países'].value_counts().to_string())

linhas no ranking: 250

primeiros lugares por país:
Países
Angola                 39
Cabo Verde             10
São Tomé e Príncipe     1


In [20]:
# conferência: cada NCM precisa ter as 5 posições, de 1 a 5
tudo_certo = True
for ncm, grupo in ranking.groupby('Código NCM'):
    if sorted(grupo['posicao'].tolist()) != [1, 2, 3, 4, 5]:
        tudo_certo = False
        print("ERRO no NCM", ncm, sorted(grupo['posicao'].tolist()))

print("todos os NCMs têm as posições 1 a 5:", tudo_certo)

todos os NCMs têm as posições 1 a 5:

 True


## 11. Exemplo de leitura do ranking

Um produto por vez, as cinco posições na ordem.

In [21]:
exemplo = '17019900'

colunas = ['posicao', 'Países', 'caso', 'peso', 'posicao_no_caso',
           'anos_ativos', 'valor_total']

print("NCM", exemplo, "-", descricoes[exemplo][:60])
print()
print(ranking[ranking['Código NCM'] == exemplo][colunas].to_string(index=False))

NCM 17019900 - Outros açúcares de cana, beterraba, sacarose químicamente pu

 posicao              Países         caso  peso  posicao_no_caso  anos_ativos  valor_total
       1              Angola continuidade    35                1           11   1075985947
       2          Cabo Verde continuidade    35                2           11     56901166
       3 São Tomé e Príncipe continuidade    35                4           11      9802227
       4        Guiné-Bissau continuidade    35                5           11      8443523
       5          Moçambique intermitente    20               19            9      2075769


## 12. Salvar

In [22]:
resultado.to_csv(PASTA_RESULTADOS / "classificacao.csv", index=False, encoding='utf-8-sig')
print("classificacao.csv:", resultado.shape)

ranking.to_csv(PASTA_RESULTADOS / "ranking.csv", index=False, encoding='utf-8-sig')
print("ranking.csv:", ranking.shape)

classificacao.csv: (250, 9)
ranking.csv: (250, 10)
